In [1]:
import os
import pickle
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('output/label_prediction_num.csv')
df.head()

,experiment_id,label,prediction
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0


In [3]:
exp_1 = df[df['experiment_id'] == 41]
exp_1

,experiment_id,label,prediction
5975,41,0,0
5976,41,0,0
5977,41,0,0
5978,41,0,0
5979,41,0,0
...,...,...,...
6074,41,0,1
6075,41,0,1
6076,41,0,1
6077,41,0,1


In [4]:
def state_space_classifier(arr, window=7):  
    state_preds = []  
    state = 'pre-void'
    terminated = False
    arr_length = len(arr)
    
    i = 0
    while i < arr_length:
        if terminated:
            state_preds.append('post-void')
            i += 1
            continue
        
        if state == 'pre-void':
            if i + window <= arr_length and all(x == 1 for x in arr[i:i+window]):
                state = 'void'
                state_preds.append(state)
                i += 1
                continue
                
        elif state == 'void':
            if i + window <= arr_length and all(x == 0 for x in arr[i:i+window]):
                state = 'post-void'
                terminated = True
                state_preds.append(state)
                i += 1
                continue
        
        state_preds.append(state)
        i += 1
    
    return state_preds

In [5]:
arr = exp_1['prediction'].values.tolist()
states = state_space_classifier(arr)
states

['pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'pre-void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 'post-void',
 '

In [6]:
import numpy as np
from hmmlearn import hmm

# Step 1: Define observed sequence from your XGBoost model
# Example: 0 = non-void, 1 = void
# observations = np.array([0, 0, 0, 1, 1, 1, 0, 0]).reshape(-1, 1)
observations = np.array(exp_1['prediction']).reshape(-1, 1)  # Use the predictions from the DataFrame
# observations = np.array(df['prediction']).reshape(-1, 1)  # Use the predictions from the DataFrame

# Step 2: Define HMM parameters
# Hidden states: 0 = pre-void, 1 = void, 2 = post-void
n_states = 3
n_observations = 2  # 0 and 1 from XGBoost

# Transition matrix: left-to-right (no jumping back)
# Rows = from-state, Cols = to-state
transmat = np.array([
    [0.8, 0.2, 0.0],  # pre-void
    [0.0, 0.8, 0.2],  # void
    [0.0, 0.0, 1.0]   # post-void (absorbing)
])

# Emission probabilities
# Each row corresponds to a hidden state:
# [P(non-void|state), P(void|state)]
emissionprob = np.array([
    [0.9, 0.1],  # pre-void mostly emits non-void
    [0.2, 0.8],  # void mostly emits void
    [0.8, 0.2]   # post-void mostly emits non-void again
])

# Start probabilities
startprob = np.array([1.0, 0.0, 0.0])  # always start in pre-void

# Step 3: Create and apply the model
model = hmm.CategoricalHMM(
    n_components=n_states,
    init_params="",
    n_iter=10
)

model.startprob_ = startprob
model.transmat_ = transmat
model.emissionprob_ = emissionprob

# Step 4: Decode hidden states
logprob, hidden_states = model.decode(observations, algorithm="viterbi")

# Step 5: Map numeric states to labels
state_map = {0: 'pre-void', 1: 'void', 2: 'post-void'}
state_sequence = [state_map[s] for s in hidden_states]

# Output
print("Binary observations:", observations.ravel().tolist())
print("Decoded HMM states:", state_sequence)

Binary observations: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1]
Decoded HMM states: ['pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'post-void', 'po

In [7]:
arr1 = np.array(states)
arr1

array(['pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void',
       'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void',
       'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void',
       'pre-void', 'pre-void', 'pre-void', 'pre-void', 'void', 'void',
       'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void',
       'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void',
       'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void',
       'void', 'void', 'void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-

In [8]:
arr2 = np.array(state_sequence)
arr2

array(['pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void',
       'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void',
       'pre-void', 'pre-void', 'pre-void', 'pre-void', 'pre-void',
       'pre-void', 'pre-void', 'pre-void', 'pre-void', 'void', 'void',
       'void', 'void', 'void', 'void', 'void', 'void', 'void', 'void',
       'void', 'void', 'void', 'void', 'void', 'void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-void', 'post-void', 'post-void', 'post-void', 'post-void',
       'post-

In [9]:
np.array_equal(arr1, arr2)  # Check if the two arrays are equal

False